In [1]:
# ============================================================
# RQ1 Reproducibility Script
# Speed profiling across emulator testing styles
# Base = instrumentation-executed + first attempt + usable verdict
#
# Reads:
#   D:\0.1-Data_Mar18_V20.0\MainDataset.csv
#
# Writes:
#   D:\0.1-Data_Mar18_V20.0\RQ1\
#       - rq1_base_counts.csv
#       - rq1_medians_and_support.csv
#       - rq1_overall_kw.csv
#       - rq1_pairwise_stats.csv
#       - rq1_summary.txt
#
# Notes:
# - Usable verdict = run_conclusion in {"success", "failure"}
# - Layer 2 uses the selected stage3 columns
# ============================================================

from __future__ import annotations

import itertools
import math
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu


# -----------------------------
# Paths
# -----------------------------
INPUT_DIR = Path(r"D:\0.1-Data_Mar18_V20.0")
INPUT_CSV = INPUT_DIR / "MainDataset.csv"
OUTPUT_DIR = INPUT_DIR / "RQ1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Study setup
# -----------------------------
STYLE_ORDER = ["Community", "Custom", "GMD", "Third-Party"]
USABLE_VERDICTS = {"success", "failure"}

OUTCOMES = {
    "Run duration": "study_run_duration_seconds",
    "Layer1 time to instrumentation envelope": "study_layer1_time_to_instrumentation_envelope_seconds",
    "Layer1 instrumentation job envelope": "study_layer1_instrumentation_job_envelope_seconds",
    "Layer1 post-instrumentation tail": "study_layer1_post_instrumentation_tail_seconds",
    "Layer2 pre-invocation": "study_pre_invocation_selected_stage3_seconds",
    "Layer2 invocation execution window": "study_invocation_execution_window_selected_stage3_seconds",
    "Layer2 post-invocation": "study_post_invocation_selected_stage3_seconds",
}


# -----------------------------
# Helper functions
# -----------------------------
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """
    Cliff's delta using rank-based Mann-Whitney relation:
    delta = 2U / (n1*n2) - 1
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan

    # Use scipy U statistic
    u_stat, _ = mannwhitneyu(x, y, alternative="two-sided")
    return (2.0 * u_stat / (len(x) * len(y))) - 1.0


def delta_magnitude(delta: float) -> str:
    """
    Romano-style rough interpretation.
    """
    if pd.isna(delta):
        return "NA"
    ad = abs(delta)
    if ad < 0.147:
        return "negligible"
    if ad < 0.33:
        return "small"
    if ad < 0.474:
        return "small-to-medium"
    return "medium-or-larger"


def epsilon_squared_from_kruskal(H: float, k: int, n: int) -> float:
    """
    Epsilon-squared for Kruskal-Wallis.
    Common form: (H - k + 1) / (n - k)
    """
    if n <= k:
        return np.nan
    return max(0.0, (H - k + 1.0) / (n - k))


def holm_adjust(pvals: List[float]) -> List[float]:
    """
    Holm correction.
    """
    m = len(pvals)
    indexed = sorted(enumerate(pvals), key=lambda t: t[1])
    adjusted = [None] * m

    running_max = 0.0
    for rank, (idx, p) in enumerate(indexed, start=1):
        adj = (m - rank + 1) * p
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


def fmt_p(p: float) -> str:
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return f"{p:.2e}"
    return f"{p:.4f}"


def fmt_num(x: float) -> str:
    if pd.isna(x):
        return "NA"
    return f"{x:.1f}"


# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(INPUT_CSV)

# -----------------------------
# Define Base under updated methodology
# -----------------------------
base = df[
    (df["controller_instru_job_count_gt0"] == True)
    & (df["controller_attempt_eq_1"] == True)
    & (df["run_conclusion"].isin(USABLE_VERDICTS))
].copy()

# Optional style scope safety
base = base[base["style"].isin(STYLE_ORDER)].copy()

# -----------------------------
# 1) Base counts
# -----------------------------
base_counts = (
    base.groupby("style", dropna=False)
    .size()
    .reindex(STYLE_ORDER, fill_value=0)
    .reset_index(name="n_base")
)

base_counts.to_csv(OUTPUT_DIR / "rq1_base_counts.csv", index=False)

# -----------------------------
# 2) Medians and support counts
# -----------------------------
median_rows = []
for outcome_label, col in OUTCOMES.items():
    for style in STYLE_ORDER:
        s = base.loc[base["style"] == style, col].dropna()
        median_rows.append(
            {
                "outcome": outcome_label,
                "column": col,
                "style": style,
                "n_non_missing": int(s.shape[0]),
                "median": float(s.median()) if len(s) else np.nan,
                "iqr": float(s.quantile(0.75) - s.quantile(0.25)) if len(s) else np.nan,
                "p95": float(s.quantile(0.95)) if len(s) else np.nan,
            }
        )

medians_df = pd.DataFrame(median_rows)
medians_df.to_csv(OUTPUT_DIR / "rq1_medians_and_support.csv", index=False)

# -----------------------------
# 3) Overall multi-style tests
# -----------------------------
overall_rows = []
for outcome_label, col in OUTCOMES.items():
    grouped = []
    valid_styles = []
    for style in STYLE_ORDER:
        s = base.loc[base["style"] == style, col].dropna().to_numpy()
        if len(s) > 0:
            grouped.append(s)
            valid_styles.append(style)

    n_total = sum(len(g) for g in grouped)
    k = len(grouped)

    if k >= 2:
        H, p = kruskal(*grouped)
        eps2 = epsilon_squared_from_kruskal(H, k, n_total)
    else:
        H, p, eps2 = np.nan, np.nan, np.nan

    overall_rows.append(
        {
            "outcome": outcome_label,
            "column": col,
            "styles_included": ", ".join(valid_styles),
            "n_total_non_missing": n_total,
            "k_groups": k,
            "kruskal_H": H,
            "p_value": p,
            "epsilon_squared": eps2,
        }
    )

overall_df = pd.DataFrame(overall_rows)
overall_df.to_csv(OUTPUT_DIR / "rq1_overall_kw.csv", index=False)

# -----------------------------
# 4) Pairwise tests
# -----------------------------
pairwise_rows = []
for outcome_label, col in OUTCOMES.items():
    raw_pvals = []
    pair_meta = []

    for s1, s2 in itertools.combinations(STYLE_ORDER, 2):
        x = base.loc[base["style"] == s1, col].dropna().to_numpy()
        y = base.loc[base["style"] == s2, col].dropna().to_numpy()

        if len(x) == 0 or len(y) == 0:
            p = np.nan
            u = np.nan
            delta = np.nan
        else:
            u, p = mannwhitneyu(x, y, alternative="two-sided")
            delta = cliffs_delta(x, y)

        raw_pvals.append(p)
        pair_meta.append((s1, s2, len(x), len(y), u, delta))

    # Holm on non-NaN only
    valid_idx = [i for i, p in enumerate(raw_pvals) if not pd.isna(p)]
    valid_p = [raw_pvals[i] for i in valid_idx]
    valid_adj = holm_adjust(valid_p)

    adj_map = {i: np.nan for i in range(len(raw_pvals))}
    for idx, adj in zip(valid_idx, valid_adj):
        adj_map[idx] = adj

    for i, ((s1, s2, n1, n2, u, delta), raw_p) in enumerate(zip(pair_meta, raw_pvals)):
        pairwise_rows.append(
            {
                "outcome": outcome_label,
                "column": col,
                "style_1": s1,
                "style_2": s2,
                "n1": n1,
                "n2": n2,
                "median_1": base.loc[base["style"] == s1, col].dropna().median(),
                "median_2": base.loc[base["style"] == s2, col].dropna().median(),
                "mannwhitney_U": u,
                "p_value_raw": raw_p,
                "p_value_holm": adj_map[i],
                "cliffs_delta": delta,
                "delta_magnitude": delta_magnitude(delta),
            }
        )

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(OUTPUT_DIR / "rq1_pairwise_stats.csv", index=False)

# -----------------------------
# 5) Printed summary
# -----------------------------
summary_lines: List[str] = []
summary_lines.append("RQ1 reproducibility summary")
summary_lines.append("=" * 72)
summary_lines.append(f"Input file: {INPUT_CSV}")
summary_lines.append(f"Base size: {len(base):,}")
summary_lines.append("")
summary_lines.append("Base counts by style:")
for _, row in base_counts.iterrows():
    summary_lines.append(f"  - {row['style']}: {row['n_base']:,}")
summary_lines.append("")

summary_lines.append("Median timings by style")
summary_lines.append("-" * 72)
for outcome_label, col in OUTCOMES.items():
    summary_lines.append(f"{outcome_label} [{col}]")
    tmp = medians_df[medians_df["outcome"] == outcome_label].copy()
    for style in STYLE_ORDER:
        r = tmp[tmp["style"] == style].iloc[0]
        summary_lines.append(
            f"  {style:12s} n={int(r['n_non_missing']):5d}  "
            f"median={fmt_num(r['median']):>8s}  "
            f"IQR={fmt_num(r['iqr']):>8s}  "
            f"P95={fmt_num(r['p95']):>8s}"
        )
    overall_row = overall_df[overall_df["outcome"] == outcome_label].iloc[0]
    summary_lines.append(
        f"  Overall Kruskal-Wallis: H={overall_row['kruskal_H']:.3f}, "
        f"p={fmt_p(overall_row['p_value'])}, "
        f"epsilon^2={overall_row['epsilon_squared']:.4f}"
    )
    summary_lines.append("")

summary_lines.append("Selected pairwise results")
summary_lines.append("-" * 72)

# Print all pairwise in compact form
for outcome_label in OUTCOMES.keys():
    summary_lines.append(f"{outcome_label}")
    tmp = pairwise_df[pairwise_df["outcome"] == outcome_label].copy()
    for _, r in tmp.iterrows():
        summary_lines.append(
            f"  {r['style_1']} vs {r['style_2']}: "
            f"medians=({fmt_num(r['median_1'])}, {fmt_num(r['median_2'])}), "
            f"p_raw={fmt_p(r['p_value_raw'])}, "
            f"p_holm={fmt_p(r['p_value_holm'])}, "
            f"delta={r['cliffs_delta']:.3f} ({r['delta_magnitude']})"
        )
    summary_lines.append("")

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(OUTPUT_DIR / "rq1_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved outputs to:", OUTPUT_DIR)

RQ1 reproducibility summary
Input file: D:\0.1-Data_Mar18_V20.0\MainDataset.csv
Base size: 7,803

Base counts by style:
  - Community: 6,980
  - Custom: 34
  - GMD: 263
  - Third-Party: 526

Median timings by style
------------------------------------------------------------------------
Run duration [study_run_duration_seconds]
  Community    n= 6980  median=   877.0  IQR=  1492.8  P95= 13601.5
  Custom       n=   34  median=  1817.0  IQR=   874.0  P95=  3247.5
  GMD          n=  263  median=  1212.0  IQR=   154.0  P95=  1483.4
  Third-Party  n=  526  median=  1864.5  IQR=  1043.5  P95=34603064.5
  Overall Kruskal-Wallis: H=302.957, p=2.28e-65, epsilon^2=0.0385

Layer1 time to instrumentation envelope [study_layer1_time_to_instrumentation_envelope_seconds]
  Community    n= 6980  median=     4.0  IQR=   208.0  P95=  1645.0
  Custom       n=   34  median=     3.0  IQR=   313.2  P95=   599.6
  GMD          n=  263  median=     3.0  IQR=     1.0  P95=     7.0
  Third-Party  n=  526  media

In [ ]:
##RQ2

In [1]:
# ============================================================
# RQ2 Reproducibility Script
# Predictability and tail risk across emulator testing styles
# Base = instrumentation-executed + first attempt + usable verdict
#
# Reads:
#   D:\0.1-Data_Mar18_V20.0\MainDataset.csv
#
# Writes:
#   D:\0.1-Data_Mar18_V20.0\RQ2\
#       - rq2_base_counts.csv
#       - rq2_dispersion_summary.csv
#       - rq2_predictability_values_long.csv
#       - rq2_overall_kw.csv
#       - rq2_pairwise_stats.csv
#       - rq2_summary.txt
#
# Main predictability measure:
#   normalized absolute deviation from each style's own median
#   NAD = abs(x - median_style) / median_style
#
# Notes:
# - Usable verdict = run_conclusion in {"success", "failure"}
# - Layer 2 uses the selected stage3 columns
# ============================================================

from __future__ import annotations

import itertools
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu


# -----------------------------
# Paths
# -----------------------------
INPUT_DIR = Path(r"D:\0.1-Data_Mar18_V20.0")
INPUT_CSV = INPUT_DIR / "MainDataset.csv"
OUTPUT_DIR = INPUT_DIR / "RQ2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Study setup
# -----------------------------
STYLE_ORDER = ["Community", "Custom", "GMD", "Third-Party"]
USABLE_VERDICTS = {"success", "failure"}

OUTCOMES = {
    "Run duration": "study_run_duration_seconds",
    "Layer1 time to instrumentation envelope": "study_layer1_time_to_instrumentation_envelope_seconds",
    "Layer1 instrumentation job envelope": "study_layer1_instrumentation_job_envelope_seconds",
    "Layer1 post-instrumentation tail": "study_layer1_post_instrumentation_tail_seconds",
    "Layer2 pre-invocation": "study_pre_invocation_selected_stage3_seconds",
    "Layer2 invocation execution window": "study_invocation_execution_window_selected_stage3_seconds",
    "Layer2 post-invocation": "study_post_invocation_selected_stage3_seconds",
}


# -----------------------------
# Helper functions
# -----------------------------
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    u_stat, _ = mannwhitneyu(x, y, alternative="two-sided")
    return (2.0 * u_stat / (len(x) * len(y))) - 1.0


def delta_magnitude(delta: float) -> str:
    if pd.isna(delta):
        return "NA"
    ad = abs(delta)
    if ad < 0.147:
        return "negligible"
    if ad < 0.33:
        return "small"
    if ad < 0.474:
        return "small-to-medium"
    return "medium-or-larger"


def epsilon_squared_from_kruskal(H: float, k: int, n: int) -> float:
    if n <= k:
        return np.nan
    return max(0.0, (H - k + 1.0) / (n - k))


def holm_adjust(pvals: List[float]) -> List[float]:
    m = len(pvals)
    indexed = sorted(enumerate(pvals), key=lambda t: t[1])
    adjusted = [None] * m
    running_max = 0.0
    for rank, (idx, p) in enumerate(indexed, start=1):
        adj = (m - rank + 1) * p
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted


def fmt_p(p: float) -> str:
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return f"{p:.2e}"
    return f"{p:.4f}"


def fmt_num(x: float) -> str:
    if pd.isna(x):
        return "NA"
    return f"{x:.3f}"


def safe_ratio(num: float, den: float) -> float:
    if pd.isna(num) or pd.isna(den) or den == 0:
        return np.nan
    return num / den


# -----------------------------
# Load data and define Base
# -----------------------------
df = pd.read_csv(INPUT_CSV)

base = df[
    (df["controller_instru_job_count_gt0"] == True)
    & (df["controller_attempt_eq_1"] == True)
    & (df["run_conclusion"].isin(USABLE_VERDICTS))
].copy()

base = base[base["style"].isin(STYLE_ORDER)].copy()

# -----------------------------
# 1) Base counts
# -----------------------------
base_counts = (
    base.groupby("style", dropna=False)
    .size()
    .reindex(STYLE_ORDER, fill_value=0)
    .reset_index(name="n_base")
)
base_counts.to_csv(OUTPUT_DIR / "rq2_base_counts.csv", index=False)

# -----------------------------
# 2) Dispersion / tail summaries
# -----------------------------
disp_rows = []
for outcome_label, col in OUTCOMES.items():
    for style in STYLE_ORDER:
        s = base.loc[base["style"] == style, col].dropna()
        median = float(s.median()) if len(s) else np.nan
        iqr = float(s.quantile(0.75) - s.quantile(0.25)) if len(s) else np.nan
        p95 = float(s.quantile(0.95)) if len(s) else np.nan
        disp_rows.append(
            {
                "outcome": outcome_label,
                "column": col,
                "style": style,
                "n_non_missing": int(s.shape[0]),
                "median": median,
                "iqr": iqr,
                "p95": p95,
                "iqr_over_median": safe_ratio(iqr, median),
                "p95_over_median": safe_ratio(p95, median),
            }
        )

disp_df = pd.DataFrame(disp_rows)
disp_df.to_csv(OUTPUT_DIR / "rq2_dispersion_summary.csv", index=False)

# -----------------------------
# 3) Build normalized absolute deviation values
# -----------------------------
nad_rows = []
for outcome_label, col in OUTCOMES.items():
    for style in STYLE_ORDER:
        s = base.loc[base["style"] == style, ["style", col]].dropna().copy()
        if s.empty:
            continue
        median_style = s[col].median()
        if median_style == 0 or pd.isna(median_style):
            # Skip zero-median normalization to avoid undefined values
            continue
        s["outcome"] = outcome_label
        s["column"] = col
        s["style_median"] = median_style
        s["nad"] = (s[col] - median_style).abs() / median_style
        nad_rows.append(s[["outcome", "column", "style", col, "style_median", "nad"]])

nad_long = pd.concat(nad_rows, ignore_index=True)
nad_long.to_csv(OUTPUT_DIR / "rq2_predictability_values_long.csv", index=False)

# -----------------------------
# 4) Overall Kruskal-Wallis on NAD
# -----------------------------
overall_rows = []
for outcome_label, col in OUTCOMES.items():
    tmp = nad_long[nad_long["outcome"] == outcome_label].copy()

    grouped = []
    valid_styles = []
    for style in STYLE_ORDER:
        s = tmp.loc[tmp["style"] == style, "nad"].dropna().to_numpy()
        if len(s) > 0:
            grouped.append(s)
            valid_styles.append(style)

    n_total = sum(len(g) for g in grouped)
    k = len(grouped)

    if k >= 2:
        H, p = kruskal(*grouped)
        eps2 = epsilon_squared_from_kruskal(H, k, n_total)
    else:
        H, p, eps2 = np.nan, np.nan, np.nan

    overall_rows.append(
        {
            "outcome": outcome_label,
            "column": col,
            "styles_included": ", ".join(valid_styles),
            "n_total_non_missing": n_total,
            "k_groups": k,
            "kruskal_H": H,
            "p_value": p,
            "epsilon_squared": eps2,
        }
    )

overall_df = pd.DataFrame(overall_rows)
overall_df.to_csv(OUTPUT_DIR / "rq2_overall_kw.csv", index=False)

# -----------------------------
# 5) Pairwise Mann-Whitney on NAD
# -----------------------------
pairwise_rows = []
for outcome_label, col in OUTCOMES.items():
    tmp = nad_long[nad_long["outcome"] == outcome_label].copy()

    raw_pvals = []
    pair_meta = []

    for s1, s2 in itertools.combinations(STYLE_ORDER, 2):
        x = tmp.loc[tmp["style"] == s1, "nad"].dropna().to_numpy()
        y = tmp.loc[tmp["style"] == s2, "nad"].dropna().to_numpy()

        if len(x) == 0 or len(y) == 0:
            p = np.nan
            u = np.nan
            delta = np.nan
            med1 = np.nan
            med2 = np.nan
        else:
            u, p = mannwhitneyu(x, y, alternative="two-sided")
            delta = cliffs_delta(x, y)
            med1 = float(np.median(x))
            med2 = float(np.median(y))

        raw_pvals.append(p)
        pair_meta.append((s1, s2, len(x), len(y), med1, med2, u, delta))

    valid_idx = [i for i, p in enumerate(raw_pvals) if not pd.isna(p)]
    valid_p = [raw_pvals[i] for i in valid_idx]
    valid_adj = holm_adjust(valid_p)

    adj_map = {i: np.nan for i in range(len(raw_pvals))}
    for idx, adj in zip(valid_idx, valid_adj):
        adj_map[idx] = adj

    for i, ((s1, s2, n1, n2, med1, med2, u, delta), raw_p) in enumerate(zip(pair_meta, raw_pvals)):
        pairwise_rows.append(
            {
                "outcome": outcome_label,
                "column": col,
                "style_1": s1,
                "style_2": s2,
                "n1": n1,
                "n2": n2,
                "median_nad_1": med1,
                "median_nad_2": med2,
                "mannwhitney_U": u,
                "p_value_raw": raw_p,
                "p_value_holm": adj_map[i],
                "cliffs_delta": delta,
                "abs_cliffs_delta": abs(delta) if not pd.isna(delta) else np.nan,
                "delta_magnitude": delta_magnitude(delta),
            }
        )

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(OUTPUT_DIR / "rq2_pairwise_stats.csv", index=False)

# -----------------------------
# 6) Summary text
# -----------------------------
summary_lines: List[str] = []
summary_lines.append("RQ2 reproducibility summary")
summary_lines.append("=" * 72)
summary_lines.append(f"Input file: {INPUT_CSV}")
summary_lines.append(f"Base size: {len(base):,}")
summary_lines.append("")
summary_lines.append("Base counts by style:")
for _, row in base_counts.iterrows():
    summary_lines.append(f"  - {row['style']}: {row['n_base']:,}")
summary_lines.append("")

summary_lines.append("Dispersion and tail summaries by style")
summary_lines.append("-" * 72)
for outcome_label, col in OUTCOMES.items():
    summary_lines.append(f"{outcome_label} [{col}]")
    tmp = disp_df[disp_df["outcome"] == outcome_label].copy()
    for style in STYLE_ORDER:
        r = tmp[tmp["style"] == style].iloc[0]
        summary_lines.append(
            f"  {style:12s} n={int(r['n_non_missing']):5d}  "
            f"median={fmt_num(r['median']):>8s}  "
            f"IQR={fmt_num(r['iqr']):>8s}  "
            f"P95={fmt_num(r['p95']):>8s}  "
            f"IQR/med={fmt_num(r['iqr_over_median']):>8s}  "
            f"P95/med={fmt_num(r['p95_over_median']):>8s}"
        )
    summary_lines.append("")

summary_lines.append("Predictability-loss (median normalized absolute deviation)")
summary_lines.append("-" * 72)
for outcome_label in OUTCOMES.keys():
    summary_lines.append(f"{outcome_label}")
    tmp = pairwise_df[pairwise_df["outcome"] == outcome_label].copy()

    # First show style medians on NAD
    med_map = (
        nad_long[nad_long["outcome"] == outcome_label]
        .groupby("style")["nad"]
        .median()
        .reindex(STYLE_ORDER)
    )
    for style, med in med_map.items():
        summary_lines.append(f"  {style:12s} median_NAD={fmt_num(med)}")

    overall_row = overall_df[overall_df["outcome"] == outcome_label].iloc[0]
    summary_lines.append(
        f"  Overall Kruskal-Wallis: H={overall_row['kruskal_H']:.3f}, "
        f"p={fmt_p(overall_row['p_value'])}, "
        f"epsilon^2={overall_row['epsilon_squared']:.4f}"
    )

    for _, r in tmp.iterrows():
        summary_lines.append(
            f"  {r['style_1']} vs {r['style_2']}: "
            f"median_NAD=({fmt_num(r['median_nad_1'])}, {fmt_num(r['median_nad_2'])}), "
            f"p_raw={fmt_p(r['p_value_raw'])}, "
            f"p_holm={fmt_p(r['p_value_holm'])}, "
            f"delta={r['cliffs_delta']:.3f} ({r['delta_magnitude']})"
        )
    summary_lines.append("")

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(OUTPUT_DIR / "rq2_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved outputs to:", OUTPUT_DIR)

RQ2 reproducibility summary
Input file: D:\0.1-Data_Mar18_V20.0\MainDataset.csv
Base size: 7,803

Base counts by style:
  - Community: 6,980
  - Custom: 34
  - GMD: 263
  - Third-Party: 526

Dispersion and tail summaries by style
------------------------------------------------------------------------
Run duration [study_run_duration_seconds]
  Community    n= 6980  median= 877.000  IQR=1492.750  P95=13601.500  IQR/med=   1.702  P95/med=  15.509
  Custom       n=   34  median=1817.000  IQR= 874.000  P95=3247.500  IQR/med=   0.481  P95/med=   1.787
  GMD          n=  263  median=1212.000  IQR= 154.000  P95=1483.400  IQR/med=   0.127  P95/med=   1.224
  Third-Party  n=  526  median=1864.500  IQR=1043.500  P95=34603064.500  IQR/med=   0.560  P95/med=18558.898

Layer1 time to instrumentation envelope [study_layer1_time_to_instrumentation_envelope_seconds]
  Community    n= 6980  median=   4.000  IQR= 208.000  P95=1645.000  IQR/med=  52.000  P95/med= 411.250
  Custom       n=   34  median= 

In [2]:
# ============================================================
# RQ3 Reproducibility Script
# Overhead composition and overhead placement across styles
# Base = instrumentation-executed + first attempt + usable verdict
#
# Reads:
#   D:\0.1-Data_Mar18_V20.0\MainDataset.csv
#
# Writes:
#   D:\0.1-Data_Mar18_V20.0\RQ3\
#       - rq3_base_counts.csv
#       - rq3_layer2_counts.csv
#       - rq3_absolute_component_summary.csv
#       - rq3_share_summary.csv
#       - rq3_overall_kw.csv
#       - rq3_pairwise_stats.csv
#       - rq3_summary.txt
#
# Notes:
# - Uses Layer 2 selected-stage3 timing columns
# - Share variables are computed as component / run_duration
# - RQ3 is restricted to Layer 2 observable Base records
# ============================================================

from __future__ import annotations

import itertools
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu


# -----------------------------
# Paths
# -----------------------------
INPUT_DIR = Path(r"D:\0.1-Data_Mar18_V20.0")
INPUT_CSV = INPUT_DIR / "MainDataset.csv"
OUTPUT_DIR = INPUT_DIR / "RQ3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Study setup
# -----------------------------
STYLE_ORDER = ["Community", "Custom", "GMD", "Third-Party"]
USABLE_VERDICTS = {"success", "failure"}

COL_RUN_DURATION = "study_run_duration_seconds"
COL_PRE = "study_pre_invocation_selected_stage3_seconds"
COL_EXEC = "study_invocation_execution_window_selected_stage3_seconds"
COL_POST = "study_post_invocation_selected_stage3_seconds"

ABS_OUTCOMES = {
    "Pre-invocation": COL_PRE,
    "Invocation execution window": COL_EXEC,
    "Post-invocation": COL_POST,
}

SHARE_OUTCOMES = {
    "Pre share": "rq3_pre_share",
    "Execution-window share": "rq3_exec_share",
    "Post share": "rq3_post_share",
}


# -----------------------------
# Helper functions
# -----------------------------
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    u_stat, _ = mannwhitneyu(x, y, alternative="two-sided")
    return (2.0 * u_stat / (len(x) * len(y))) - 1.0


def delta_magnitude(delta: float) -> str:
    if pd.isna(delta):
        return "NA"
    ad = abs(delta)
    if ad < 0.147:
        return "negligible"
    if ad < 0.33:
        return "small"
    if ad < 0.474:
        return "small-to-medium"
    return "medium-or-larger"


def epsilon_squared_from_kruskal(H: float, k: int, n: int) -> float:
    if n <= k:
        return np.nan
    return max(0.0, (H - k + 1.0) / (n - k))


def holm_adjust(pvals: List[float]) -> List[float]:
    m = len(pvals)
    indexed = sorted(enumerate(pvals), key=lambda t: t[1])
    adjusted = [None] * m
    running_max = 0.0
    for rank, (idx, p) in enumerate(indexed, start=1):
        adj = (m - rank + 1) * p
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted


def fmt_p(p: float) -> str:
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return f"{p:.2e}"
    return f"{p:.4f}"


def fmt_num(x: float) -> str:
    if pd.isna(x):
        return "NA"
    return f"{x:.3f}"


# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(INPUT_CSV)

# -----------------------------
# Define Base
# -----------------------------
base = df[
    (df["controller_instru_job_count_gt0"] == True)
    & (df["controller_attempt_eq_1"] == True)
    & (df["run_conclusion"].isin(USABLE_VERDICTS))
].copy()

base = base[base["style"].isin(STYLE_ORDER)].copy()

# -----------------------------
# Base counts by style
# -----------------------------
base_counts = (
    base.groupby("style", dropna=False)
    .size()
    .reindex(STYLE_ORDER, fill_value=0)
    .reset_index(name="n_base")
)
base_counts.to_csv(OUTPUT_DIR / "rq3_base_counts.csv", index=False)

# -----------------------------
# Restrict to Layer 2 observable subset
# -----------------------------
layer2 = base[
    base[COL_RUN_DURATION].notna()
    & base[COL_PRE].notna()
    & base[COL_EXEC].notna()
    & base[COL_POST].notna()
    & (base[COL_RUN_DURATION] > 0)
].copy()

# Compute shares
layer2["rq3_pre_share"] = layer2[COL_PRE] / layer2[COL_RUN_DURATION]
layer2["rq3_exec_share"] = layer2[COL_EXEC] / layer2[COL_RUN_DURATION]
layer2["rq3_post_share"] = layer2[COL_POST] / layer2[COL_RUN_DURATION]

# Layer 2 counts by style
layer2_counts = (
    layer2.groupby("style", dropna=False)
    .size()
    .reindex(STYLE_ORDER, fill_value=0)
    .reset_index(name="n_layer2")
)
layer2_counts.to_csv(OUTPUT_DIR / "rq3_layer2_counts.csv", index=False)

# -----------------------------
# Absolute component medians
# -----------------------------
abs_rows = []
for outcome_label, col in ABS_OUTCOMES.items():
    for style in STYLE_ORDER:
        s = layer2.loc[layer2["style"] == style, col].dropna()
        abs_rows.append(
            {
                "outcome": outcome_label,
                "column": col,
                "style": style,
                "n_non_missing": int(s.shape[0]),
                "median": float(s.median()) if len(s) else np.nan,
                "iqr": float(s.quantile(0.75) - s.quantile(0.25)) if len(s) else np.nan,
                "p95": float(s.quantile(0.95)) if len(s) else np.nan,
            }
        )

abs_df = pd.DataFrame(abs_rows)
abs_df.to_csv(OUTPUT_DIR / "rq3_absolute_component_summary.csv", index=False)

# -----------------------------
# Share summaries
# -----------------------------
share_rows = []
for outcome_label, col in SHARE_OUTCOMES.items():
    for style in STYLE_ORDER:
        s = layer2.loc[layer2["style"] == style, col].dropna()
        share_rows.append(
            {
                "outcome": outcome_label,
                "column": col,
                "style": style,
                "n_non_missing": int(s.shape[0]),
                "median": float(s.median()) if len(s) else np.nan,
                "iqr": float(s.quantile(0.75) - s.quantile(0.25)) if len(s) else np.nan,
                "p95": float(s.quantile(0.95)) if len(s) else np.nan,
            }
        )

share_df = pd.DataFrame(share_rows)
share_df.to_csv(OUTPUT_DIR / "rq3_share_summary.csv", index=False)

# -----------------------------
# Overall Kruskal-Wallis on shares
# -----------------------------
overall_rows = []
for outcome_label, col in SHARE_OUTCOMES.items():
    grouped = []
    valid_styles = []
    for style in STYLE_ORDER:
        s = layer2.loc[layer2["style"] == style, col].dropna().to_numpy()
        if len(s) > 0:
            grouped.append(s)
            valid_styles.append(style)

    n_total = sum(len(g) for g in grouped)
    k = len(grouped)

    if k >= 2:
        H, p = kruskal(*grouped)
        eps2 = epsilon_squared_from_kruskal(H, k, n_total)
    else:
        H, p, eps2 = np.nan, np.nan, np.nan

    overall_rows.append(
        {
            "outcome": outcome_label,
            "column": col,
            "styles_included": ", ".join(valid_styles),
            "n_total_non_missing": n_total,
            "k_groups": k,
            "kruskal_H": H,
            "p_value": p,
            "epsilon_squared": eps2,
        }
    )

overall_df = pd.DataFrame(overall_rows)
overall_df.to_csv(OUTPUT_DIR / "rq3_overall_kw.csv", index=False)

# -----------------------------
# Pairwise tests on shares
# -----------------------------
pairwise_rows = []
for outcome_label, col in SHARE_OUTCOMES.items():
    raw_pvals = []
    pair_meta = []

    for s1, s2 in itertools.combinations(STYLE_ORDER, 2):
        x = layer2.loc[layer2["style"] == s1, col].dropna().to_numpy()
        y = layer2.loc[layer2["style"] == s2, col].dropna().to_numpy()

        if len(x) == 0 or len(y) == 0:
            p = np.nan
            u = np.nan
            delta = np.nan
            med1 = np.nan
            med2 = np.nan
        else:
            u, p = mannwhitneyu(x, y, alternative="two-sided")
            delta = cliffs_delta(x, y)
            med1 = float(np.median(x))
            med2 = float(np.median(y))

        raw_pvals.append(p)
        pair_meta.append((s1, s2, len(x), len(y), med1, med2, u, delta))

    valid_idx = [i for i, p in enumerate(raw_pvals) if not pd.isna(p)]
    valid_p = [raw_pvals[i] for i in valid_idx]
    valid_adj = holm_adjust(valid_p)

    adj_map = {i: np.nan for i in range(len(raw_pvals))}
    for idx, adj in zip(valid_idx, valid_adj):
        adj_map[idx] = adj

    for i, ((s1, s2, n1, n2, med1, med2, u, delta), raw_p) in enumerate(zip(pair_meta, raw_pvals)):
        pairwise_rows.append(
            {
                "outcome": outcome_label,
                "column": col,
                "style_1": s1,
                "style_2": s2,
                "n1": n1,
                "n2": n2,
                "median_1": med1,
                "median_2": med2,
                "mannwhitney_U": u,
                "p_value_raw": raw_p,
                "p_value_holm": adj_map[i],
                "cliffs_delta": delta,
                "abs_cliffs_delta": abs(delta) if not pd.isna(delta) else np.nan,
                "delta_magnitude": delta_magnitude(delta),
            }
        )

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(OUTPUT_DIR / "rq3_pairwise_stats.csv", index=False)

# -----------------------------
# Summary text
# -----------------------------
summary_lines: List[str] = []
summary_lines.append("RQ3 reproducibility summary")
summary_lines.append("=" * 72)
summary_lines.append(f"Input file: {INPUT_CSV}")
summary_lines.append(f"Base size: {len(base):,}")
summary_lines.append(f"Layer 2 observable Base size: {len(layer2):,}")
summary_lines.append("")

summary_lines.append("Base counts by style:")
for _, row in base_counts.iterrows():
    summary_lines.append(f"  - {row['style']}: {row['n_base']:,}")
summary_lines.append("")

summary_lines.append("Layer 2 observable counts by style:")
for _, row in layer2_counts.iterrows():
    summary_lines.append(f"  - {row['style']}: {row['n_layer2']:,}")
summary_lines.append("")

summary_lines.append("Absolute Layer 2 component medians")
summary_lines.append("-" * 72)
for outcome_label, col in ABS_OUTCOMES.items():
    summary_lines.append(f"{outcome_label} [{col}]")
    tmp = abs_df[abs_df["outcome"] == outcome_label].copy()
    for style in STYLE_ORDER:
        r = tmp[tmp["style"] == style].iloc[0]
        summary_lines.append(
            f"  {style:12s} n={int(r['n_non_missing']):5d}  "
            f"median={fmt_num(r['median']):>8s}  "
            f"IQR={fmt_num(r['iqr']):>8s}  "
            f"P95={fmt_num(r['p95']):>8s}"
        )
    summary_lines.append("")

summary_lines.append("Layer 2 placement shares")
summary_lines.append("-" * 72)
for outcome_label, col in SHARE_OUTCOMES.items():
    summary_lines.append(f"{outcome_label} [{col}]")
    tmp = share_df[share_df["outcome"] == outcome_label].copy()
    for style in STYLE_ORDER:
        r = tmp[tmp["style"] == style].iloc[0]
        summary_lines.append(
            f"  {style:12s} n={int(r['n_non_missing']):5d}  "
            f"median_share={fmt_num(r['median']):>8s}  "
            f"IQR={fmt_num(r['iqr']):>8s}  "
            f"P95={fmt_num(r['p95']):>8s}"
        )

    overall_row = overall_df[overall_df["outcome"] == outcome_label].iloc[0]
    summary_lines.append(
        f"  Overall Kruskal-Wallis: H={overall_row['kruskal_H']:.3f}, "
        f"p={fmt_p(overall_row['p_value'])}, "
        f"epsilon^2={overall_row['epsilon_squared']:.4f}"
    )

    tmp2 = pairwise_df[pairwise_df["outcome"] == outcome_label].copy()
    for _, r in tmp2.iterrows():
        summary_lines.append(
            f"  {r['style_1']} vs {r['style_2']}: "
            f"medians=({fmt_num(r['median_1'])}, {fmt_num(r['median_2'])}), "
            f"p_raw={fmt_p(r['p_value_raw'])}, "
            f"p_holm={fmt_p(r['p_value_holm'])}, "
            f"delta={r['cliffs_delta']:.3f} ({r['delta_magnitude']})"
        )
    summary_lines.append("")

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(OUTPUT_DIR / "rq3_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved outputs to:", OUTPUT_DIR)

RQ3 reproducibility summary
Input file: D:\0.1-Data_Mar18_V20.0\MainDataset.csv
Base size: 7,803
Layer 2 observable Base size: 5,116

Base counts by style:
  - Community: 6,980
  - Custom: 34
  - GMD: 263
  - Third-Party: 526

Layer 2 observable counts by style:
  - Community: 4,838
  - Custom: 30
  - GMD: 134
  - Third-Party: 114

Absolute Layer 2 component medians
------------------------------------------------------------------------
Pre-invocation [study_pre_invocation_selected_stage3_seconds]
  Community    n= 4838  median= 181.000  IQR= 825.750  P95=1852.150
  Custom       n=   30  median= 131.500  IQR= 364.750  P95= 702.700
  GMD          n=  134  median=  79.500  IQR=  16.000  P95= 260.100
  Third-Party  n=  114  median= 561.000  IQR=  67.750  P95=1343.500

Invocation execution window [study_invocation_execution_window_selected_stage3_seconds]
  Community    n= 4838  median= 392.000  IQR= 719.000  P95=2390.150
  Custom       n=   30  median= 726.000  IQR= 927.000  P95=1643.750

In [5]:
# ============================================================
# RQ4 Reproducibility Script
# Deployment context and run-level verdict usability
#
# RQ4 regime:
#   instrumentation-executed + first attempt
#   (all outcomes retained; no usable-verdict filter)
#
# Reads:
#   D:\0.1-Data_Mar18_V20.0\MainDataset.csv
#
# Writes:
#   D:\0.1-Data_Mar18_V20.0\RQ4\
#       - rq4_first_attempt_counts.csv
#       - rq4_usable_verdict_by_style.csv
#       - rq4_success_among_usable_by_style.csv
#       - rq4_event_by_style_counts.csv
#       - rq4_overall_tests.csv
#       - rq4_pairwise_tests.csv
#       - rq4_within_style_event_conditioned_success.csv
#       - rq4_summary.txt
#
# Notes:
# - usable verdict = run_conclusion in {"success", "failure"}
# - success rate is computed among usable verdicts only
# - event column assumed to be: event
# - robust to degenerate 2x2 event tables
# ============================================================

from __future__ import annotations

import itertools
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact


# -----------------------------
# Paths
# -----------------------------
INPUT_DIR = Path(r"D:\0.1-Data_Mar18_V20.0")
INPUT_CSV = INPUT_DIR / "MainDataset.csv"
OUTPUT_DIR = INPUT_DIR / "RQ4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Study setup
# -----------------------------
STYLE_ORDER = ["Community", "Custom", "GMD", "Third-Party"]
USABLE_VERDICTS = {"success", "failure"}
SUCCESS = "success"
FAILURE = "failure"

# Adjust this only if your dataset uses a different event column name.
EVENT_COL = "event"
OTHER_EVENT_LABEL = "other"

MAIN_EVENTS = ["push", "pull_request", "schedule", "workflow_dispatch"]


# -----------------------------
# Helper functions
# -----------------------------
def cramers_v_from_table(table: np.ndarray) -> float:
    """
    Cramer's V from contingency table.
    Returns NaN for degenerate tables.
    """
    table = np.asarray(table, dtype=float)

    if table.size == 0:
        return np.nan
    if table.ndim != 2:
        return np.nan
    if table.sum() == 0:
        return np.nan

    # Degenerate table: any row/col sum is zero
    if np.any(table.sum(axis=0) == 0) or np.any(table.sum(axis=1) == 0):
        return np.nan

    try:
        chi2, _, _, _ = chi2_contingency(table)
    except ValueError:
        return np.nan

    n = table.sum()
    r, c = table.shape
    denom = min(r - 1, c - 1)
    if denom <= 0:
        return np.nan

    return np.sqrt(chi2 / (n * denom))


def choose_test_2x2(table_2x2: np.ndarray) -> Tuple[str, float]:
    """
    Use Fisher for sparse/degenerate 2x2 tables, chi-square otherwise.
    """
    table_2x2 = np.asarray(table_2x2)

    if table_2x2.shape != (2, 2):
        raise ValueError("choose_test_2x2 expects a 2x2 table")

    # Degenerate row/column
    if np.any(table_2x2.sum(axis=0) == 0) or np.any(table_2x2.sum(axis=1) == 0):
        try:
            _, p = fisher_exact(table_2x2)
            return "fisher", p
        except Exception:
            return "degenerate", np.nan

    # Sparse table
    if (table_2x2 < 5).any():
        try:
            _, p = fisher_exact(table_2x2)
            return "fisher", p
        except Exception:
            return "degenerate", np.nan

    try:
        chi2, p, _, _ = chi2_contingency(table_2x2)
        return "chi2", p
    except ValueError:
        try:
            _, p = fisher_exact(table_2x2)
            return "fisher", p
        except Exception:
            return "degenerate", np.nan


def fmt_p(p: float) -> str:
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return f"{p:.2e}"
    return f"{p:.4f}"


def wilson_ci(k: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    """
    Wilson score interval for a proportion.
    """
    if n == 0:
        return (np.nan, np.nan)
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n) / denom
    return (center - half, center + half)


def simplify_event(x: object) -> str:
    """
    Keep common event names readable, collapse blanks/rare values to 'other'.
    """
    if pd.isna(x):
        return OTHER_EVENT_LABEL
    s = str(x).strip()
    if not s:
        return OTHER_EVENT_LABEL

    keep = {
        "push",
        "pull_request",
        "schedule",
        "workflow_dispatch",
        "pull_request_target",
        "release",
    }
    return s if s in keep else OTHER_EVENT_LABEL


# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(INPUT_CSV)

if EVENT_COL not in df.columns:
    raise KeyError(
        f"Expected event column '{EVENT_COL}' not found in MainDataset.csv. "
        f"Available columns include: {list(df.columns[:20])} ..."
    )

# -----------------------------
# Define RQ4 regime
# instrumentation-executed + first attempt
# all conclusions retained
# -----------------------------
rq4 = df[
    (df["controller_instru_job_count_gt0"] == True)
    & (df["controller_attempt_eq_1"] == True)
].copy()

rq4 = rq4[rq4["style"].isin(STYLE_ORDER)].copy()

rq4["event_clean"] = rq4[EVENT_COL].apply(simplify_event)
rq4["is_usable_verdict"] = rq4["run_conclusion"].isin(USABLE_VERDICTS)
rq4["is_success"] = rq4["run_conclusion"].eq(SUCCESS)
rq4["is_failure"] = rq4["run_conclusion"].eq(FAILURE)


# -----------------------------
# 1) First-attempt counts by style
# -----------------------------
first_attempt_counts = (
    rq4.groupby("style", dropna=False)
    .size()
    .reindex(STYLE_ORDER, fill_value=0)
    .reset_index(name="n_first_attempt")
)
first_attempt_counts.to_csv(OUTPUT_DIR / "rq4_first_attempt_counts.csv", index=False)


# -----------------------------
# 2) Usable verdict by style
# -----------------------------
usable_rows = []
for style in STYLE_ORDER:
    sub = rq4[rq4["style"] == style]
    n_total = len(sub)
    n_success = int(sub["is_success"].sum())
    n_failure = int(sub["is_failure"].sum())
    n_usable = int(sub["is_usable_verdict"].sum())
    n_nonusable = n_total - n_usable
    n_cancelled = int((sub["run_conclusion"] == "cancelled").sum())
    n_missing = int(sub["run_conclusion"].isna().sum())
    rate = n_usable / n_total if n_total else np.nan
    lo, hi = wilson_ci(n_usable, n_total)

    usable_rows.append(
        {
            "style": style,
            "n_total": n_total,
            "n_success": n_success,
            "n_failure": n_failure,
            "n_usable": n_usable,
            "n_nonusable": n_nonusable,
            "n_cancelled": n_cancelled,
            "n_missing_conclusion": n_missing,
            "usable_verdict_rate": rate,
            "usable_verdict_rate_ci_low": lo,
            "usable_verdict_rate_ci_high": hi,
        }
    )

usable_df = pd.DataFrame(usable_rows)
usable_df.to_csv(OUTPUT_DIR / "rq4_usable_verdict_by_style.csv", index=False)


# -----------------------------
# 3) Success among usable verdicts
# -----------------------------
success_rows = []
usable_only = rq4[rq4["is_usable_verdict"]].copy()

for style in STYLE_ORDER:
    sub = usable_only[usable_only["style"] == style]
    n_usable = len(sub)
    n_success = int(sub["is_success"].sum())
    n_failure = int(sub["is_failure"].sum())
    success_rate = n_success / n_usable if n_usable else np.nan
    lo, hi = wilson_ci(n_success, n_usable)

    success_rows.append(
        {
            "style": style,
            "n_usable": n_usable,
            "n_success": n_success,
            "n_failure": n_failure,
            "success_rate_among_usable": success_rate,
            "success_rate_ci_low": lo,
            "success_rate_ci_high": hi,
        }
    )

success_df = pd.DataFrame(success_rows)
success_df.to_csv(OUTPUT_DIR / "rq4_success_among_usable_by_style.csv", index=False)


# -----------------------------
# 4) Event distribution by style
# -----------------------------
event_counts = pd.crosstab(
    rq4["style"],
    rq4["event_clean"],
    dropna=False,
).reindex(index=STYLE_ORDER, fill_value=0)

event_counts.to_csv(OUTPUT_DIR / "rq4_event_by_style_counts.csv")


# -----------------------------
# 5) Overall tests
# -----------------------------
overall_test_rows = []

# 5a) style x usable verdict
table_style_usable = pd.crosstab(
    rq4["style"],
    rq4["is_usable_verdict"],
    dropna=False,
).reindex(index=STYLE_ORDER, columns=[False, True], fill_value=0)

chi2, p, _, _ = chi2_contingency(table_style_usable.values)
overall_test_rows.append(
    {
        "test_name": "style_x_usable_verdict",
        "table_shape": str(table_style_usable.shape),
        "n_total": int(table_style_usable.values.sum()),
        "chi2": chi2,
        "p_value": p,
        "effect_size": cramers_v_from_table(table_style_usable.values),
        "effect_name": "cramers_v",
    }
)

# 5b) style x success/failure among usable only
table_style_success = pd.crosstab(
    usable_only["style"],
    usable_only["run_conclusion"],
    dropna=False,
).reindex(index=STYLE_ORDER, columns=[SUCCESS, FAILURE], fill_value=0)

chi2, p, _, _ = chi2_contingency(table_style_success.values)
overall_test_rows.append(
    {
        "test_name": "style_x_success_failure_among_usable",
        "table_shape": str(table_style_success.shape),
        "n_total": int(table_style_success.values.sum()),
        "chi2": chi2,
        "p_value": p,
        "effect_size": cramers_v_from_table(table_style_success.values),
        "effect_name": "cramers_v",
    }
)

# 5c) style x event
table_style_event = pd.crosstab(
    rq4["style"],
    rq4["event_clean"],
    dropna=False,
).reindex(index=STYLE_ORDER, fill_value=0)

chi2, p, _, _ = chi2_contingency(table_style_event.values)
overall_test_rows.append(
    {
        "test_name": "style_x_event",
        "table_shape": str(table_style_event.shape),
        "n_total": int(table_style_event.values.sum()),
        "chi2": chi2,
        "p_value": p,
        "effect_size": cramers_v_from_table(table_style_event.values),
        "effect_name": "cramers_v",
    }
)

overall_tests_df = pd.DataFrame(overall_test_rows)
overall_tests_df.to_csv(OUTPUT_DIR / "rq4_overall_tests.csv", index=False)


# -----------------------------
# 6) Pairwise tests
# -----------------------------
pairwise_rows = []

# 6a) Pairwise style comparisons on usable verdict rate
for s1, s2 in itertools.combinations(STYLE_ORDER, 2):
    sub = rq4[rq4["style"].isin([s1, s2])].copy()
    table = pd.crosstab(sub["style"], sub["is_usable_verdict"]).reindex(
        index=[s1, s2], columns=[False, True], fill_value=0
    )

    if table.values.sum() == 0:
        continue

    test_name, p = choose_test_2x2(table.values)
    pairwise_rows.append(
        {
            "comparison_family": "pairwise_style_usable_verdict",
            "group_1": s1,
            "group_2": s2,
            "test_used": test_name,
            "p_value": p,
            "effect_size": cramers_v_from_table(table.values),
            "effect_name": "cramers_v",
            "table_json": str(table.to_dict()),
        }
    )

# 6b) Pairwise style comparisons on success/failure among usable only
for s1, s2 in itertools.combinations(STYLE_ORDER, 2):
    sub = usable_only[usable_only["style"].isin([s1, s2])].copy()
    table = pd.crosstab(sub["style"], sub["run_conclusion"]).reindex(
        index=[s1, s2], columns=[SUCCESS, FAILURE], fill_value=0
    )

    if table.values.sum() == 0:
        continue

    test_name, p = choose_test_2x2(table.values)
    pairwise_rows.append(
        {
            "comparison_family": "pairwise_style_success_failure_among_usable",
            "group_1": s1,
            "group_2": s2,
            "test_used": test_name,
            "p_value": p,
            "effect_size": cramers_v_from_table(table.values),
            "effect_name": "cramers_v",
            "table_json": str(table.to_dict()),
        }
    )

# 6c) Pairwise style comparisons on selected event presence
for event_name in MAIN_EVENTS:
    indicator_col = f"is_event_{event_name}"
    rq4[indicator_col] = rq4["event_clean"].eq(event_name)

    for s1, s2 in itertools.combinations(STYLE_ORDER, 2):
        sub = rq4[rq4["style"].isin([s1, s2])].copy()
        table = pd.crosstab(sub["style"], sub[indicator_col]).reindex(
            index=[s1, s2], columns=[False, True], fill_value=0
        )

        if table.values.sum() == 0 or table.shape != (2, 2):
            continue

        test_name, p = choose_test_2x2(table.values)
        pairwise_rows.append(
            {
                "comparison_family": f"pairwise_style_event_{event_name}",
                "group_1": s1,
                "group_2": s2,
                "test_used": test_name,
                "p_value": p,
                "effect_size": cramers_v_from_table(table.values),
                "effect_name": "cramers_v",
                "table_json": str(table.to_dict()),
            }
        )

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(OUTPUT_DIR / "rq4_pairwise_tests.csv", index=False)


# -----------------------------
# 7) Within-style event-conditioned success-rate tests
# Among usable verdicts only
# -----------------------------
within_style_rows = []

for style in STYLE_ORDER:
    sub = usable_only[usable_only["style"] == style].copy()
    if sub.empty:
        continue

    event_counts_style = sub["event_clean"].value_counts()
    style_events = [e for e, n in event_counts_style.items() if n > 0]

    for e1, e2 in itertools.combinations(style_events, 2):
        sub2 = sub[sub["event_clean"].isin([e1, e2])].copy()
        table = pd.crosstab(sub2["event_clean"], sub2["run_conclusion"]).reindex(
            index=[e1, e2], columns=[SUCCESS, FAILURE], fill_value=0
        )

        if table.shape != (2, 2) or table.values.sum() == 0:
            continue

        test_name, p = choose_test_2x2(table.values)

        n1 = int(table.loc[e1].sum())
        n2 = int(table.loc[e2].sum())
        sr1 = table.loc[e1, SUCCESS] / n1 if n1 else np.nan
        sr2 = table.loc[e2, SUCCESS] / n2 if n2 else np.nan

        within_style_rows.append(
            {
                "style": style,
                "event_1": e1,
                "event_2": e2,
                "n_event_1": n1,
                "n_event_2": n2,
                "success_rate_event_1": sr1,
                "success_rate_event_2": sr2,
                "test_used": test_name,
                "p_value": p,
                "effect_size": cramers_v_from_table(table.values),
                "effect_name": "cramers_v",
                "table_json": str(table.to_dict()),
            }
        )

within_style_df = pd.DataFrame(within_style_rows)
within_style_df.to_csv(
    OUTPUT_DIR / "rq4_within_style_event_conditioned_success.csv", index=False
)


# -----------------------------
# 8) Summary text
# -----------------------------
summary_lines: List[str] = []
summary_lines.append("RQ4 reproducibility summary")
summary_lines.append("=" * 72)
summary_lines.append(f"Input file: {INPUT_CSV}")
summary_lines.append(f"RQ4 regime size (first-attempt instrumentation-executed): {len(rq4):,}")
summary_lines.append("")

summary_lines.append("First-attempt counts by style:")
for _, row in first_attempt_counts.iterrows():
    summary_lines.append(f"  - {row['style']}: {row['n_first_attempt']:,}")
summary_lines.append("")

summary_lines.append("Usable verdict by style:")
for _, row in usable_df.iterrows():
    summary_lines.append(
        f"  - {row['style']}: usable={row['n_usable']}/{row['n_total']} "
        f"({row['usable_verdict_rate']*100:.1f}%), "
        f"success={row['n_success']}, failure={row['n_failure']}, "
        f"cancelled={row['n_cancelled']}, missing={row['n_missing_conclusion']}"
    )
summary_lines.append("")

summary_lines.append("Success rate among usable verdicts:")
for _, row in success_df.iterrows():
    summary_lines.append(
        f"  - {row['style']}: success={row['n_success']}/{row['n_usable']} "
        f"({row['success_rate_among_usable']*100:.1f}%)"
    )
summary_lines.append("")

summary_lines.append("Overall tests:")
for _, row in overall_tests_df.iterrows():
    summary_lines.append(
        f"  - {row['test_name']}: chi2={row['chi2']:.3f}, "
        f"p={fmt_p(row['p_value'])}, "
        f"Cramer's V={row['effect_size']:.4f}"
    )
summary_lines.append("")

summary_lines.append("Event distribution by style:")
event_prop = event_counts.div(event_counts.sum(axis=1), axis=0).fillna(0)
for style in STYLE_ORDER:
    if style not in event_prop.index:
        continue
    summary_lines.append(f"  {style}:")
    for ev, prop in event_prop.loc[style].sort_values(ascending=False).items():
        if prop > 0:
            summary_lines.append(f"    - {ev}: {prop*100:.1f}%")
summary_lines.append("")

summary_lines.append("Selected within-style event-conditioned success-rate tests:")
if within_style_df.empty:
    summary_lines.append("  - No within-style event-conditioned contrasts available.")
else:
    for _, row in within_style_df.sort_values(["style", "p_value"]).head(20).iterrows():
        sr1 = row["success_rate_event_1"] * 100 if pd.notna(row["success_rate_event_1"]) else np.nan
        sr2 = row["success_rate_event_2"] * 100 if pd.notna(row["success_rate_event_2"]) else np.nan
        summary_lines.append(
            f"  - {row['style']}: {row['event_1']} vs {row['event_2']} "
            f"(success rates {sr1:.1f}% vs {sr2:.1f}%), "
            f"{row['test_used']} p={fmt_p(row['p_value'])}, "
            f"Cramer's V={row['effect_size']:.4f}"
        )

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(OUTPUT_DIR / "rq4_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved outputs to:", OUTPUT_DIR)

RQ4 reproducibility summary
Input file: D:\0.1-Data_Mar18_V20.0\MainDataset.csv
RQ4 regime size (first-attempt instrumentation-executed): 8,666

First-attempt counts by style:
  - Community: 7,826
  - Custom: 39
  - GMD: 265
  - Third-Party: 536

Usable verdict by style:
  - Community: usable=6980/7826 (89.2%), success=5523, failure=1457, cancelled=841, missing=5
  - Custom: usable=34/39 (87.2%), success=8, failure=26, cancelled=5, missing=0
  - GMD: usable=263/265 (99.2%), success=258, failure=5, cancelled=2, missing=0
  - Third-Party: usable=526/536 (98.1%), success=87, failure=439, cancelled=10, missing=0

Success rate among usable verdicts:
  - Community: success=5523/6980 (79.1%)
  - Custom: success=8/34 (23.5%)
  - GMD: success=258/263 (98.1%)
  - Third-Party: success=87/526 (16.5%)

Overall tests:
  - style_x_usable_verdict: chi2=70.871, p=2.78e-15, Cramer's V=0.0904
  - style_x_success_failure_among_usable: chi2=1154.040, p=6.87e-250, Cramer's V=0.3846
  - style_x_event: chi2=4

In [ ]:
#robustness check

In [1]:
# -*- coding: utf-8 -*-
"""
Compute Table XI:
Tier-2 paired coarsened-family statistical support
for the selected key observations.

Reads:
    C:\Android Mobile App\ICST2026_Ext\MainDataset.csv

Writes:
    C:\Android Mobile App\ICST2026_Ext\2.0-RQ_Support_Robustness_Check\
        - table_xi_tier2_stats.csv
        - table_xi_tier2_stats.tex
        - table_xi_tier2_stats_readme.txt

Notes
-----
- RQ1–RQ3 robustness checks are run under the Base timing regime.
- RQ4 robustness checks are run under the first-attempt outcome regime.
- Obs. 4.2 is additionally conditioned on usable verdicts (success/failure).
- Tier 2 families are built from eligible exact signatures using:
    * same runner OS bucket
    * adjacent-or-equal executed job-count bucket
    * adjacent-or-equal executed step-count bucket
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional
import math
import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu, chi2_contingency


# ============================================================
# CONFIG
# ============================================================
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
IN_MAIN = BASE_DIR / "MainDataset.csv"
OUT_DIR = BASE_DIR / "2.0-RQ_Support_Robustness_Check"

OUT_CSV = OUT_DIR / "table_xi_tier2_stats.csv"
OUT_TEX = OUT_DIR / "table_xi_tier2_stats.tex"
OUT_README = OUT_DIR / "table_xi_tier2_stats_readme.txt"

MIN_SIGNATURE_TOTAL_N = 80
MIN_SIGNATURE_STYLE_N = 15
MIN_SIGNATURE_USABLE_STYLES = 2

STYLE_ORDER = ["Community", "Custom", "GMD", "Third-Party"]

SIGNATURE_HASH_COL = "study_signature_hash"
RUNNER_OS_COL = "study_runner_os_bucket"
JOB_BUCKET_COL = "study_job_count_exec_bucket"
STEP_BUCKET_COL = "study_step_count_exec_bucket"

# RQ1–RQ3 timing metrics
RUN_DURATION_COL = "study_run_duration_seconds"
PRE_COL = "study_pre_invocation_selected_stage3_seconds"
EXEC_COL = "study_invocation_execution_window_selected_stage3_seconds"
POST_COL = "study_post_invocation_selected_stage3_seconds"

# Regime flags
BASE_FLAG_COL = "Base"
ROBUST_FLAG_COL = "Robust"
FIRST_ATTEMPT_FLAG_COL = "controller_attempt_eq_1"

# RQ4 fields
STYLE_COL = "style"
CONCLUSION_COL = "run_conclusion"
EVENT_COL = "event"

JOB_BUCKET_ADJ = {
    "1": {"1", "2_3"},
    "2_3": {"1", "2_3", "4_6"},
    "4_6": {"2_3", "4_6", ">6"},
    ">6": {"4_6", ">6"},
}

STEP_BUCKET_ADJ = {
    "<=20": {"<=20", "21_40"},
    "21_40": {"<=20", "21_40", "41_80"},
    "41_80": {"21_40", "41_80", ">80"},
    ">80": {"41_80", ">80"},
}


# ============================================================
# HELPERS
# ============================================================
def norm_bool(series: pd.Series) -> pd.Series:
    """Normalize mixed bool-like columns into pandas boolean dtype."""
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")

    s = series.copy()
    s = s.replace({1: True, 0: False})
    s = s.astype(str).str.strip().str.lower()

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
        "y": True,
        "n": False,
    }
    out = s.map(mapping)
    return out.astype("boolean")


def blank_to_na(series: pd.Series) -> pd.Series:
    return series.replace(r"^\s*$", np.nan, regex=True)


def is_known_bucket(x) -> bool:
    return pd.notna(x) and str(x).strip() != "" and str(x).strip().lower() != "unknown"


def cliffs_delta(x: pd.Series, y: pd.Series) -> float:
    """
    Compute Cliff's delta from Mann-Whitney U.
    """
    xv = pd.to_numeric(x, errors="coerce").dropna().to_numpy(dtype=float)
    yv = pd.to_numeric(y, errors="coerce").dropna().to_numpy(dtype=float)

    if len(xv) == 0 or len(yv) == 0:
        return float("nan")

    u = mannwhitneyu(xv, yv, alternative="two-sided", method="asymptotic").statistic
    return float((2.0 * u) / (len(xv) * len(yv)) - 1.0)


def cramers_v_from_table(table: pd.DataFrame) -> float:
    chi2, _, _, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    r, k = table.shape
    if n == 0 or min(r - 1, k - 1) <= 0:
        return float("nan")
    return float(math.sqrt(chi2 / (n * min(r - 1, k - 1))))


def scientific_or_decimal(x: float, decimals: int = 3) -> str:
    if pd.isna(x):
        return "--"
    if x == 0:
        return "0"
    if abs(x) < 1e-3 or abs(x) >= 1e4:
        return f"{x:.2e}"
    return f"{x:.{decimals}f}"


def choose_main_anchor(eligible_df: pd.DataFrame) -> str:
    """
    Choose the main anchor used for RQ1–RQ3:
    prefer an eligible signature with Community + GMD + Third-Party support.
    Break ties by larger total_n.
    """
    cand = eligible_df[
        (eligible_df.get("Community", 0) > 0)
        & (eligible_df.get("GMD", 0) > 0)
        & (eligible_df.get("Third-Party", 0) > 0)
    ].copy()

    if not cand.empty:
        cand = cand.sort_values(["total_n", "usable_style_count"], ascending=[False, False])
        return str(cand.iloc[0][SIGNATURE_HASH_COL])

    # Fallback
    eligible_sorted = eligible_df.sort_values(["total_n", "usable_style_count"], ascending=[False, False])
    return str(eligible_sorted.iloc[0][SIGNATURE_HASH_COL])


def build_family_mask(df: pd.DataFrame, sig_row: pd.Series) -> pd.Series:
    """
    Tier 2 family:
      same OS bucket,
      adjacent-or-equal job-count bucket,
      adjacent-or-equal step-count bucket,
      requires nonblank strict signature hash.
    """
    os_bucket = str(sig_row[RUNNER_OS_COL]).strip()
    job_bucket = str(sig_row[JOB_BUCKET_COL]).strip()
    step_bucket = str(sig_row[STEP_BUCKET_COL]).strip()

    job_family = JOB_BUCKET_ADJ.get(job_bucket, {job_bucket})
    step_family = STEP_BUCKET_ADJ.get(step_bucket, {step_bucket})

    return (
        df[SIGNATURE_HASH_COL].notna()
        & (df[RUNNER_OS_COL].astype(str).str.strip() == os_bucket)
        & (df[JOB_BUCKET_COL].astype(str).str.strip().isin(job_family))
        & (df[STEP_BUCKET_COL].astype(str).str.strip().isin(step_family))
    )


def add_norm_abs_dev_by_style(
    df: pd.DataFrame,
    subset_mask: pd.Series,
    source_col: str,
    out_col: str,
) -> pd.Series:
    """
    For the given subset, compute |x - median_style| / median_style within style.
    """
    out = pd.Series(np.nan, index=df.index, dtype="float64")
    sub = df.loc[subset_mask, [STYLE_COL, source_col]].copy()
    sub[source_col] = pd.to_numeric(sub[source_col], errors="coerce")

    medians = sub.groupby(STYLE_COL, dropna=True)[source_col].median()

    for style_name, med in medians.items():
        if pd.isna(med) or med == 0:
            continue
        mask = subset_mask & df[STYLE_COL].eq(style_name)
        vals = pd.to_numeric(df.loc[mask, source_col], errors="coerce")
        out.loc[mask] = (vals - med).abs() / med

    return out


def build_mwu_row(
    df: pd.DataFrame,
    obs: str,
    family_label: str,
    metric_label: str,
    metric_col: str,
    style1: str,
    style2: str,
    subset_mask: pd.Series,
    interpretation: str,
) -> Dict[str, object]:
    sub = df.loc[subset_mask].copy()

    x = pd.to_numeric(sub.loc[sub[STYLE_COL] == style1, metric_col], errors="coerce").dropna()
    y = pd.to_numeric(sub.loc[sub[STYLE_COL] == style2, metric_col], errors="coerce").dropna()

    n1, n2 = int(len(x)), int(len(y))
    med1 = float(np.median(x)) if n1 else float("nan")
    med2 = float(np.median(y)) if n2 else float("nan")

    if n1 and n2:
        p = float(mannwhitneyu(x, y, alternative="two-sided", method="asymptotic").pvalue)
        delta = cliffs_delta(x, y)
    else:
        p = float("nan")
        delta = float("nan")

    return {
        "Obs.": obs,
        "Tier 2 family": family_label,
        "Metric": metric_label,
        "Contrast": f"{style1} vs. {style2}",
        "n1": n1,
        "n2": n2,
        "Median1": med1,
        "Median2": med2,
        "Test": "MWU",
        "p_value": p,
        "Effect": delta,
        "Interpretation": interpretation,
    }


def build_chi_row(
    df: pd.DataFrame,
    obs: str,
    family_label: str,
    metric_label: str,
    row_var: str,
    col_var: str,
    subset_mask: pd.Series,
    interpretation: str,
) -> Dict[str, object]:
    sub = df.loc[subset_mask].copy()

    if row_var == EVENT_COL:
        sub = sub[sub[EVENT_COL].notna()].copy()

    if col_var == "success_flag":
        sub = sub[sub["success_flag"].notna()].copy()

    ct = pd.crosstab(sub[row_var], sub[col_var])

    if ct.empty or ct.shape[0] < 2 or ct.shape[1] < 2:
        p = float("nan")
        effect = float("nan")
    else:
        _, p, _, _ = chi2_contingency(ct)
        effect = cramers_v_from_table(ct)

    return {
        "Obs.": obs,
        "Tier 2 family": family_label,
        "Metric": metric_label,
        "Contrast": f"{row_var} × {col_var}",
        "n1": int(ct.to_numpy().sum()) if not ct.empty else 0,
        "n2": int(ct.shape[0]) if not ct.empty else 0,
        "Median1": float("nan"),
        "Median2": float("nan"),
        "Test": "chi2",
        "p_value": float(p),
        "Effect": float(effect),
        "Interpretation": interpretation,
    }


def format_median_pair(row: pd.Series) -> str:
    if pd.isna(row["Median1"]) and pd.isna(row["Median2"]):
        return "--"
    return f"{scientific_or_decimal(row['Median1'])} / {scientific_or_decimal(row['Median2'])}"


def format_effect(row: pd.Series) -> str:
    if row["Test"] == "MWU":
        return f"$p = {scientific_or_decimal(row['p_value'])}$, $\\delta={row['Effect']:.3f}$" if pd.notna(row["p_value"]) and pd.notna(row["Effect"]) else "--"
    return f"$p = {scientific_or_decimal(row['p_value'])}$, $V={row['Effect']:.3f}$" if pd.notna(row["p_value"]) and pd.notna(row["Effect"]) else "--"


def latex_escape(s: str) -> str:
    if s is None:
        return ""
    replacements = {
        "\\": r"\textbackslash{}",
        "_": r"\_",
        "%": r"\%",
        "&": r"\&",
        "#": r"\#",
        "$": r"\$",
    }
    out = str(s)
    for k, v in replacements.items():
        out = out.replace(k, v)
    return out


def render_table_xi_latex(table_df: pd.DataFrame) -> str:
    lines = []
    lines.append(r"\begin{table*}[t]")
    lines.append(r"\centering")
    lines.append(r"\caption{Tier~2 paired coarsened-family statistical support for the selected key observations.}")
    lines.append(r"\label{tab:key_obs_tier2_tests}")
    lines.append(r"\footnotesize")
    lines.append(r"\setlength{\tabcolsep}{4pt}")
    lines.append(r"\begin{tabular}{p{0.8cm} p{2.5cm} p{2.0cm} p{1.5cm} p{1.7cm} p{1.3cm} p{1.8cm} p{3.2cm}}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Obs.} & \textbf{Tier~2 family} & \textbf{Metric} & \textbf{Contrast} & \textbf{$n_1$ / $n_2$} & \textbf{Median$_1$ / Median$_2$} & \textbf{$p$ / effect} & \textbf{Interpretation} \\")
    lines.append(r"\midrule")

    for _, row in table_df.iterrows():
        family = latex_escape(str(row["Tier 2 family"]))
        metric = latex_escape(str(row["Metric"]))
        contrast = latex_escape(str(row["Contrast"]))
        n_pair = f"{int(row['n1'])} / {int(row['n2'])}"
        med_pair = latex_escape(format_median_pair(row))
        peff = format_effect(row)
        interp = latex_escape(str(row["Interpretation"]))

        lines.append(
            f"{latex_escape(str(row['Obs.']))} & "
            f"{family} & "
            f"{metric} & "
            f"{contrast} & "
            f"{n_pair} & "
            f"{med_pair} & "
            f"{peff} & "
            f"{interp} \\\\"
        )

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\vspace{2pt}")
    lines.append(r"\begin{flushleft}")
    lines.append(
        r"\footnotesize "
        r"\textit{Note:} Tier~2 uses the paired coarsened family derived from the observation’s Tier~1 anchor. "
        r"For RQ1--RQ3, the robustness check is conducted under the Base timing regime. "
        r"For RQ4, the robustness check is conducted under the first-attempt outcome regime; "
        r"Obs.~4.2 is additionally conditioned on usable verdicts. "
        r"MWU = Mann--Whitney; $\delta$ = Cliff’s delta; $V$ = Cramer’s $V$."
    )
    lines.append(r"\end{flushleft}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)


# ============================================================
# MAIN
# ============================================================
def main() -> None:
    if not IN_MAIN.exists():
        raise FileNotFoundError(f"Input file not found: {IN_MAIN}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(IN_MAIN, low_memory=False)

    # Basic normalization
    required_cols = [
        SIGNATURE_HASH_COL,
        RUNNER_OS_COL,
        JOB_BUCKET_COL,
        STEP_BUCKET_COL,
        STYLE_COL,
        RUN_DURATION_COL,
        PRE_COL,
        EXEC_COL,
        POST_COL,
        BASE_FLAG_COL,
        ROBUST_FLAG_COL,
        FIRST_ATTEMPT_FLAG_COL,
        CONCLUSION_COL,
        EVENT_COL,
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"MainDataset is missing required columns: {missing}")

    for c in [BASE_FLAG_COL, ROBUST_FLAG_COL, FIRST_ATTEMPT_FLAG_COL]:
        df[c] = norm_bool(df[c])

    for c in [SIGNATURE_HASH_COL, RUNNER_OS_COL, JOB_BUCKET_COL, STEP_BUCKET_COL, STYLE_COL, CONCLUSION_COL, EVENT_COL]:
        df[c] = blank_to_na(df[c]).astype("object")

    # --------------------------------------------------------
    # Detect eligible exact signatures from robust pool
    # --------------------------------------------------------
    robust_pool = df.loc[df[ROBUST_FLAG_COL].fillna(False) & df[SIGNATURE_HASH_COL].notna()].copy()

    sig_counts = (
        robust_pool.groupby([SIGNATURE_HASH_COL, STYLE_COL], dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )

    sig_wide = sig_counts.pivot_table(
        index=SIGNATURE_HASH_COL,
        columns=STYLE_COL,
        values="n",
        fill_value=0,
    ).reset_index()

    for s in STYLE_ORDER:
        if s not in sig_wide.columns:
            sig_wide[s] = 0

    sig_wide["total_n"] = sig_wide[STYLE_ORDER].sum(axis=1)
    sig_wide["usable_style_count"] = (sig_wide[STYLE_ORDER] >= MIN_SIGNATURE_STYLE_N).sum(axis=1)

    sig_meta = (
        robust_pool[[SIGNATURE_HASH_COL, RUNNER_OS_COL, JOB_BUCKET_COL, STEP_BUCKET_COL]]
        .drop_duplicates(subset=[SIGNATURE_HASH_COL])
        .copy()
    )

    eligible = sig_wide.merge(sig_meta, on=SIGNATURE_HASH_COL, how="left")
    eligible = eligible[
        (eligible["total_n"] >= MIN_SIGNATURE_TOTAL_N)
        & (eligible["usable_style_count"] >= MIN_SIGNATURE_USABLE_STYLES)
    ].copy()

    if eligible.empty:
        raise ValueError("No eligible exact signatures were detected.")

    anchor_main = choose_main_anchor(eligible)
    other_anchors = [str(x) for x in eligible[SIGNATURE_HASH_COL].astype(str).tolist() if str(x) != anchor_main]

    if not other_anchors:
        raise ValueError("Only one eligible exact signature found; split-anchor RQ4 rows need two.")
    anchor_other = other_anchors[0]

    # Signature rows for family construction
    eligible_indexed = eligible.set_index(SIGNATURE_HASH_COL, drop=False)
    family_main = build_family_mask(df, eligible_indexed.loc[anchor_main])
    family_other = build_family_mask(df, eligible_indexed.loc[anchor_other])

    family_main_label = f"Paired family of {anchor_main}"
    family_other_label = f"Paired family of {anchor_other}"

    # --------------------------------------------------------
    # Regimes
    # --------------------------------------------------------
    base_mask = df[BASE_FLAG_COL].fillna(False)
    first_attempt_mask = df[FIRST_ATTEMPT_FLAG_COL].fillna(False)
    usable_mask = df[CONCLUSION_COL].isin(["success", "failure"])
    df["success_flag"] = np.where(df[CONCLUSION_COL].eq("success"), True,
                           np.where(df[CONCLUSION_COL].eq("failure"), False, np.nan))

    # --------------------------------------------------------
    # Derived Tier 2 subset metrics for RQ2 and RQ3
    # --------------------------------------------------------
    tier2_main_mask = base_mask & family_main

    df["norm_abs_dev_run_duration_t2_main"] = add_norm_abs_dev_by_style(
        df, tier2_main_mask, RUN_DURATION_COL, "norm_abs_dev_run_duration_t2_main"
    )
    df["norm_abs_dev_exec_window_t2_main"] = add_norm_abs_dev_by_style(
        df, tier2_main_mask, EXEC_COL, "norm_abs_dev_exec_window_t2_main"
    )

    l2_total = (
        pd.to_numeric(df[PRE_COL], errors="coerce")
        + pd.to_numeric(df[EXEC_COL], errors="coerce")
        + pd.to_numeric(df[POST_COL], errors="coerce")
    )
    df["l2_pre_share"] = pd.to_numeric(df[PRE_COL], errors="coerce") / l2_total
    df["l2_exec_share"] = pd.to_numeric(df[EXEC_COL], errors="coerce") / l2_total
    df["l2_post_share"] = pd.to_numeric(df[POST_COL], errors="coerce") / l2_total

    # --------------------------------------------------------
    # Build Table XI rows
    # --------------------------------------------------------
    rows: List[Dict[str, object]] = []

    # RQ1 (Base + family_main)
    rows.append(build_mwu_row(
        df, "1.1", family_main_label, "Run duration", RUN_DURATION_COL,
        "Community", "GMD", tier2_main_mask,
        "Broadened support for faster Community"
    ))
    rows.append(build_mwu_row(
        df, "1.1", family_main_label, "L2 exec. window", EXEC_COL,
        "Community", "Third-Party", tier2_main_mask,
        "Strong completion-side support"
    ))

    rows.append(build_mwu_row(
        df, "1.2", family_main_label, "L2 pre-invocation", PRE_COL,
        "GMD", "Third-Party", tier2_main_mask,
        "Fast-entry remains clear vs. Third-Party"
    ))
    rows.append(build_mwu_row(
        df, "1.2", family_main_label, "L2 pre-invocation", PRE_COL,
        "Community", "GMD", tier2_main_mask,
        "Entry-side support weakens in Tier 2"
    ))

    rows.append(build_mwu_row(
        df, "1.3", family_main_label, "Run duration", RUN_DURATION_COL,
        "Third-Party", "Community", tier2_main_mask,
        "Strong slow-path support preserved"
    ))
    rows.append(build_mwu_row(
        df, "1.3", family_main_label, "L2 execution window", EXEC_COL,
        "Third-Party", "GMD", tier2_main_mask,
        "Sustained-execution gap remains visible"
    ))

    # RQ2 (Base + family_main)
    rows.append(build_mwu_row(
        df, "2.1", family_main_label, "Norm. abs. deviation (run duration)",
        "norm_abs_dev_run_duration_t2_main",
        "GMD", "Community", tier2_main_mask,
        "Strong Tier 2 predictability support"
    ))
    rows.append(build_mwu_row(
        df, "2.1", family_main_label, "Norm. abs. deviation (L2 exec. window)",
        "norm_abs_dev_exec_window_t2_main",
        "GMD", "Third-Party", tier2_main_mask,
        "Predictability signal remains visible"
    ))

    rows.append(build_mwu_row(
        df, "2.2", family_main_label, "Norm. abs. deviation (run duration)",
        "norm_abs_dev_run_duration_t2_main",
        "Community", "GMD", tier2_main_mask,
        "Tradeoff preserved in broadened family"
    ))

    # RQ3 (Base + family_main)
    rows.append(build_mwu_row(
        df, "3.1", family_main_label, "L2 execution share", "l2_exec_share",
        "GMD", "Community", tier2_main_mask,
        "Execution-centric separation remains strong"
    ))
    rows.append(build_mwu_row(
        df, "3.1", family_main_label, "L2 post-invocation share", "l2_post_share",
        "Community", "GMD", tier2_main_mask,
        "Residual-tail contrast preserved"
    ))

    rows.append(build_mwu_row(
        df, "3.2", family_main_label, "L2 pre-invocation share", "l2_pre_share",
        "Third-Party", "GMD", tier2_main_mask,
        "Heavy-entry support remains strong"
    ))
    rows.append(build_mwu_row(
        df, "3.2", family_main_label, "L2 execution share", "l2_exec_share",
        "Third-Party", "Community", tier2_main_mask,
        "Heavy-execution pattern preserved"
    ))

    # RQ4 (first-attempt regime; 4.2 conditioned on usable verdicts)
    # Split paired families -> one row for each family so the result is fully filled and reproducible
    rows.append(build_chi_row(
        df, "4.2", family_main_label,
        "Success among usable verdicts", STYLE_COL, "success_flag",
        first_attempt_mask & usable_mask & family_main,
        "Main-family support for success-rate separation"
    ))
    rows.append(build_chi_row(
        df, "4.2", family_other_label,
        "Success among usable verdicts", STYLE_COL, "success_flag",
        first_attempt_mask & usable_mask & family_other,
        "Secondary-family support for success-rate separation"
    ))

    rows.append(build_chi_row(
        df, "4.3", family_main_label,
        "Trigger event distribution", STYLE_COL, EVENT_COL,
        first_attempt_mask & family_main,
        "Deployment-context separation remains strong in the main family"
    ))
    rows.append(build_chi_row(
        df, "4.3", family_other_label,
        "Trigger event distribution", STYLE_COL, EVENT_COL,
        first_attempt_mask & family_other,
        "Deployment-context separation remains strong in the secondary family"
    ))

    table_xi = pd.DataFrame(rows)

    # Friendly column order
    table_xi = table_xi[
        [
            "Obs.",
            "Tier 2 family",
            "Metric",
            "Contrast",
            "n1",
            "n2",
            "Median1",
            "Median2",
            "Test",
            "p_value",
            "Effect",
            "Interpretation",
        ]
    ].copy()

    # Save raw CSV
    table_xi.to_csv(OUT_CSV, index=False)

    # Save LaTeX
    tex = render_table_xi_latex(table_xi)
    OUT_TEX.write_text(tex, encoding="utf-8")

    # Save README
    readme = f"""Table XI generation completed.

Input:
  {IN_MAIN}

Outputs:
  {OUT_CSV}
  {OUT_TEX}

Detected eligible exact signatures:
  Main anchor:   {anchor_main}
  Other anchor:  {anchor_other}

Tier 2 family rule:
  - same runner OS bucket
  - adjacent-or-equal executed job-count bucket
  - adjacent-or-equal executed step-count bucket

Regimes:
  - RQ1–RQ3: Base timing regime
  - RQ4: first-attempt outcome regime
  - Obs. 4.2: first-attempt + usable verdict subset
"""
    OUT_README.write_text(readme, encoding="utf-8")

    print("Done.")
    print(f"CSV saved to: {OUT_CSV}")
    print(f"LaTeX saved to: {OUT_TEX}")
    print(f"README saved to: {OUT_README}")
    print("\nDetected anchors:")
    print(f"  main  = {anchor_main}")
    print(f"  other = {anchor_other}")
    print("\nPreview:")
    print(table_xi.to_string(index=False))


if __name__ == "__main__":
    main()

Done.
CSV saved to: C:\Android Mobile App\ICST2026_Ext\2.0-RQ_Support_Robustness_Check\table_xi_tier2_stats.csv
LaTeX saved to: C:\Android Mobile App\ICST2026_Ext\2.0-RQ_Support_Robustness_Check\table_xi_tier2_stats.tex
README saved to: C:\Android Mobile App\ICST2026_Ext\2.0-RQ_Support_Robustness_Check\table_xi_tier2_stats_readme.txt

Detected anchors:
  main  = 88f32b360855c277
  other = 0bc0e933a2435166

Preview:
Obs.                     Tier 2 family                                 Metric                  Contrast   n1   n2     Median1     Median2 Test       p_value    Effect                                                       Interpretation
 1.1 Paired family of 88f32b360855c277                           Run duration         Community vs. GMD 3094  131  737.000000 1158.000000  MWU  3.622369e-26 -0.545047                               Broadened support for faster Community
 1.1 Paired family of 88f32b360855c277                        L2 exec. window Community vs. Third-Party 2853 